# Tutorial 4: the variational quantum eigensolver (Qiskit)

*Near-term Quantum Computing 2026-2027, tutorial 4.*

In Lecture 4, we turned the adiabatic circuit into a variational one, the Hamiltonian variational
ansatz (HVA), whose angles a classical optimiser chooses to minimise the measured energy. In this
tutorial, we first run the adiabatic circuit itself, and then this variational quantum eigensolver (VQE) for the TFIM chain of the lecture,
with every energy estimated from shots, as on a quantum computer. We compare two optimisers by
what they cost in shots, and look at what gate noise does to the result.

In this tutorial you will learn to:

1. build parametrised circuits in Qiskit, and write the energy as a function of their angles;
2. minimise it with an optimiser from scipy, and read and check its result;
3. see how the accuracy of the adiabatic circuit depends on its total time and its step length;
4. compute gradients with the parameter-shift rule, and compare optimisers by their total cost
   in shots;
5. find conceptual mistakes in a VQE run and its report.

**How this tutorial works.** As in the previous tutorials:

- **Part A** (about 40 minutes): we code together. Type the code yourself in the empty cells.
- **Part B**: exercises, at home, alone or in pairs. Exercises 1, 2 and 3 are the core; 4 and 5
  go further.
- Questions marked **(paper)** are for pen and paper: answer them *before* running the code.

**Conventions**, as in Lecture 4: the transverse-field Ising model (TFIM) on an open chain,
$H = -J\sum_i Z_iZ_{i+1} - h\sum_i X_i$, with $J = 1$ and $h = 0.5$ (Tutorial 3 used $h = 1$). We
use $n = 4$ qubits instead of the 8 of the notes, so that every optimisation takes seconds. The
qubits are numbered from 0, as in Qiskit. As in Tutorial 3, an energy estimate with "$N$ shots"
uses $N$ shots in each of the two measurement settings, so $2N$ shots in all.

Documentation: [Qiskit API reference](https://quantum.cloud.ibm.com/docs/en/api/qiskit),
[`ParameterVector`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.ParameterVector),
[`scipy.optimize.minimize`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)
and its [COBYLA method](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-cobyla.html).
In a notebook, a cell with `minimize?` shows the documentation of `minimize`, and the same works
for any function.

## Setup

In Google Colab, run the next cell first to install the packages. If you work on your own
computer, install the same packages in your Python environment instead.

In [ ]:
%pip install -q "qiskit~=2.5.2" "qiskit-aer~=0.18.0" "scipy>=1.16" pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

simulator = AerSimulator()
J, h = 1.0, 0.5

The next cell is given: the functions of Tutorial 3 that we reuse. `tfim(n_qubits, J, h)` builds the
Hamiltonian. `measure_energy(circuit, shots, J, h)` estimates $\langle H\rangle$ of the state that
`circuit` prepares, from `shots` shots in each of the two settings, and returns the estimate and
its standard error; it uses `signs` and `group_values`.

In [ ]:
def tfim(n_qubits, J, h):
    """The TFIM on an open chain of n_qubits qubits, as a SparsePauliOp."""
    terms = []
    for i in range(n_qubits - 1):                      # bonds
        terms.append(("ZZ", [i, i + 1], -J))
    for i in range(n_qubits):                          # fields
        terms.append(("X", [i], -h))
    return SparsePauliOp.from_sparse_list(terms, num_qubits=n_qubits)


def signs(bitstring):
    """The outcomes of one shot as signs: signs(bitstring)[q] is +1 or -1 for qubit q."""
    return [1 - 2 * int(b) for b in bitstring[::-1]]   # reversed: Qiskit writes qubit 0 last


def group_values(counts, setting, J, h):
    """One value per shot: s_Z if setting is "Z", s_X if setting is "X"."""
    values = []
    for bitstring, count in counts.items():
        spins = signs(bitstring)
        n_qubits = len(spins)
        if setting == "Z":
            total = 0
            for i in range(n_qubits - 1):
                total += spins[i] * spins[i + 1]
            value = -J * total
        else:
            value = -h * sum(spins)
        values += [value] * count
    return np.array(values)


def measure_energy(state_circuit, shots, J, h):
    """Estimate <H> of the TFIM from `shots` shots in each of the two settings.

    Returns the estimate and its standard error."""
    n_qubits = state_circuit.num_qubits
    energy, variance = 0, 0
    for setting in ["Z", "X"]:
        circuit = state_circuit.copy()
        if setting == "X":
            circuit.h(range(n_qubits))
        circuit.measure_all()
        counts = simulator.run(circuit, shots=shots).result().get_counts()
        values = group_values(counts, setting, J, h)
        energy += values.mean()
        variance += values.var() / shots
    return energy, np.sqrt(variance)

Our chain has `n_qubits = 4` qubits. Its ground-state energy $E_0$ is the lowest eigenvalue of the matrix of
$H$ (exact diagonalisation, as in Tutorial 3). A VQE is meant for problems where $E_0$ is not
known; on a small instance, we compute it to check the method (notes, section ["The variational
quantum eigensolver"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-vqe-loop)).

In [ ]:
n_qubits = 4
hamiltonian = tfim(n_qubits, J, h)
E0 = np.linalg.eigvalsh(hamiltonian.to_matrix())[0]
print("E0 =", round(E0, 4))

## Part A: coding together

### A1. Parametrised circuits

The HVA with $p$ layers (notes, section ["The Hamiltonian variational ansatz"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-hva)) starts from
$|{+}\cdots{+}\rangle$ and applies, in layer $k$, $R_{ZZ}(\gamma_k)$ on every bond and
$R_X(\beta_k)$ on every qubit: a Trotter step of Tutorial 3 with free angles. Its $2p$ angles are
$\theta = (\gamma_1, \beta_1, \dots, \gamma_p, \beta_p)$.

An optimiser tries many values of these angles. Instead of building a new circuit for each, we
build the circuit once with *parameters*, symbols that stand for angles we choose later.
`ParameterVector("theta", m)` creates $m$ of them, `theta[0]`, ..., `theta[m - 1]`, and we use
them wherever a gate needs an angle, for example `circuit.rx(theta[1], 0)`; the same parameter
can appear in several gates. `circuit.assign_parameters(values)` then returns a copy of the
circuit with the numbers in `values` in place of `theta[0]`, `theta[1]`, ..., in this order.

Complete `hva_ansatz`, with the bonds in two layers as in `trotter_step` of Tutorial 3: `gamma` is
the angle of all the bond gates of layer `k`, and `beta` the angle of all its field gates.

In [ ]:
from qiskit.circuit import ParameterVector


def hva_ansatz(n_qubits, n_layers):
    """The HVA with n_layers layers on n_qubits qubits; its parameters are (γ_1, β_1, ..., γ_p, β_p)."""
    theta = ParameterVector("theta", 2 * n_layers)
    circuit = QuantumCircuit(n_qubits)
    circuit.h(range(n_qubits))                          # |+...+>
    for k in range(n_layers):
        gamma = theta[2 * k]
        beta = theta[2 * k + 1]
        # YOUR CODE HERE
    return circuit


print(hva_ansatz(n_qubits, 1).draw())

We use $p = 3$ layers, `n_layers = 3`: `ansatz` is the circuit that we optimise in the rest of this tutorial.

In [ ]:
n_layers = 3
ansatz = hva_ansatz(n_qubits, n_layers)
print("number of parameters:", ansatz.num_parameters)

**The adiabatic schedule** is one point of the ansatz, and our starting point (notes, section
["The Hamiltonian variational ansatz"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-hva)). With $p$ Trotter steps of length $\delta = T/p$, layer $k$
has $\gamma_k = -2s_kJ\delta$ and $\beta_k = -2h\delta$, where $s_k = (k - \frac12)/p$ is the
middle of step $k = 1, \dots, p$.

**(paper)** What are the six angles for $p = 3$ and $T = 3$? Then run the next cell, where the
given function `schedule` computes them. (Python counts `k` from 0, so there the middle of step
`k` is `(k + 0.5) / n_layers`.)

In [ ]:
def schedule(n_layers, total_time):
    """The angles (γ_1, β_1, ..., γ_p, β_p) of the adiabatic schedule.

    n_layers Trotter steps up to the time total_time."""
    dt = total_time / n_layers
    angles = []
    for k in range(n_layers):
        s_k = (k + 0.5) / n_layers                        # the middle of step k
        angles += [-2 * s_k * J * dt, -2 * h * dt]
    return np.array(angles)


start = schedule(n_layers, 3.0)
print("schedule angles:", np.round(start, 3))

*Your answer:*

**The energy as a function of the angles.** In simulation, we compute the energy of the prepared
state exactly, as in Tutorial 3. Complete `energy_exact`: assign the angles to `ansatz`, and
compute $\langle H\rangle$ of its `Statevector`.

**(paper)** What is the energy at $\theta = 0$, where every gate of a layer is the identity?

In [ ]:
def energy_exact(angles):
    """The exact energy of the state that `ansatz` prepares with these angles (simulation only)."""
    # YOUR CODE HERE


print("theta = 0: E =", round(energy_exact(np.zeros(2 * n_layers)), 4))
print("schedule:  E =", round(energy_exact(start), 4))
print("E0 =", round(E0, 4))

*Your answer:*

### A2. Minimising with scipy

`minimize(f, x0, method="COBYLA")`, from `scipy.optimize`, minimises a function `f` of an array of
numbers, starting from the array `x0`. It calls `f` many times; each call is an *evaluation*.
COBYLA uses only the values of `f` (notes, section ["Optimising with shots"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-optimisers)). `minimize` returns an
object, `result`, whose attributes hold the answer:

- `result.x`: the best angles found;
- `result.fun`: the value of `f` recorded for them (for COBYLA, the lowest value it has seen);
- `result.nfev`: the number of evaluations;
- `result.message`: why the optimiser stopped.

First, with exact energies:

In [ ]:
from scipy.optimize import minimize

# result_exact: minimise energy_exact with COBYLA, starting from `start`; print the message,
# the number of evaluations, the angles found, and result_exact.fun - E0
# YOUR CODE HERE

**With shots.** On a quantum computer, every evaluation is an estimate from shots. Complete
`energy_estimate`, which returns the estimate of `measure_energy` for the given angles, and also
keeps a record of the run: it adds every estimate to the list `history`. Afterwards, `history`
shows what the optimiser saw, and `2 * shots * len(history)` is the number of shots it used.

`minimize` passes only the angles to `f`; further arguments go in a tuple `args`:
`minimize(f, x0, args=(a, b))` calls `f(angles, a, b)`. Then run COBYLA with 1000 shots per
setting.

In [ ]:
def energy_estimate(angles, shots, history):
    """An estimate of the energy from `shots` shots per setting; the estimate is added to `history`."""
    # YOUR CODE HERE


history = []
# result: minimise energy_estimate with COBYLA from `start`, with 1000 shots per setting and `history`
# YOUR CODE HERE
print(result.message)
print("evaluations:", result.nfev, "  shots:", 2 * 1000 * len(history))
print("result.fun - E0 =", round(result.fun - E0, 3))
print("exact energy at result.x, minus E0:", round(energy_exact(result.x) - E0, 3))

The plotting code is given: `plt.plot(history, ".")` draws one dot per estimate, against its
position in the list, and `plt.axhline(y)` draws a horizontal line at `y`.

In [ ]:
plt.plot(history, ".", label="estimates seen by COBYLA")
plt.axhline(energy_exact(result.x), color="tab:orange", label="exact energy at result.x")
plt.axhline(E0, color="black", linestyle="--", label="E0")
plt.xlabel("evaluation")
plt.ylabel("energy")
plt.legend()
plt.show()

**Think.** With exact energies, COBYLA needed about 250 evaluations and ended 0.003 above $E_0$.
With shots, it stopped after about 60, much further from $E_0$, although its message is the same.
And `result.fun` is lower than the exact energy at `result.x`. Why? Exercises 2 and 3 come back to
these points.

*Your answer:*

### Summary: from Lecture 4 to code

| Lecture 4 | Qiskit and scipy |
| --- | --- |
| a circuit with parameters $\theta$ | `theta = ParameterVector("theta", m)`, then `circuit.rx(theta[0], q)`, ... |
| the circuit at given angles | `circuit.assign_parameters(angles)` |
| minimise $E(\theta)$ from $\theta_0$ | `result = minimize(f, x0, method="COBYLA")` |
| further arguments of $f$ | `minimize(f, x0, args=(shots, history), method="COBYLA")` |
| at most $m$ evaluations | `minimize(..., options={"maxiter": m})` |
| what the optimiser found | `result.x`, `result.fun`, `result.nfev`, `result.message` |

## Part B: exercises

Exercises 1, 2 and 3 are the core of this tutorial; 4 and 5 go further. Use the functions from
Part A. Every run with shots is random, so your numbers will differ a little from those quoted in
the text.

### Exercise 1 (core): the adiabatic circuit, without an optimiser

Before we optimise any angles, we run the adiabatic circuit itself (notes, section ["Adiabatic
circuits"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-adiabatic-circuits)): Trotter steps of length $\delta$ along the path
$H(s) = -sJ\sum_i Z_iZ_{i+1} - h\sum_i X_i$, from $|{+}\cdots{+}\rangle$, up to the total time
$T$. With `steps = round(total_time / delta)`, this is `hva_ansatz(n_qubits, steps)` with the
angles `schedule(steps, total_time)` of Part A.

In [ ]:
# The problem, defined again so that this exercise stands alone (as in Part A)
n_qubits = 4
hamiltonian = tfim(n_qubits, J, h)
eigenvalues, eigenvectors = np.linalg.eigh(hamiltonian.to_matrix())
E0 = eigenvalues[0]
ground_state = Statevector(eigenvectors[:, 0])     # the eigenvector of the lowest eigenvalue
print("E0 =", round(E0, 4))

**(a) (code)** For the step lengths $\delta = 0.25$, $0.5$ and $1$, and the total times
$T = 1, 2, \dots, 16$, compute the fidelity $|\langle E_0|\psi(T)\rangle|^2$ of the final state
with the ground state, and its exact energy minus $E_0$. `state_fidelity(state, ground_state)` is
the fidelity of two `Statevector`s, and `state.expectation_value(hamiltonian).real` the energy, as
in `energy_exact`. The plotting code is given: `plt.subplots(1, 2)` makes two plots side by side,
`ax_fidelity` and `ax_energy`.

In [ ]:
from qiskit.quantum_info import state_fidelity

total_times = np.arange(1, 17)                    # T = 1, 2, ..., 16

fig, (ax_fidelity, ax_energy) = plt.subplots(1, 2, figsize=(11, 4))
for delta in [0.25, 0.5, 1.0]:
    fidelities = []                               # one per total time
    errors = []                                   # the exact energy minus E0, one per total time
    for total_time in total_times:
        steps = round(total_time / delta)
        # the adiabatic circuit with these steps up to total_time; append its fidelity and E - E0
        # YOUR CODE HERE
    ax_fidelity.plot(total_times, fidelities, "o-", label="δ = " + str(delta))
    ax_energy.semilogy(total_times, errors, "o-", label="δ = " + str(delta))
ax_fidelity.set_xlabel("total time T")
ax_fidelity.set_ylabel("fidelity with the ground state")
ax_energy.set_xlabel("total time T")
ax_energy.set_ylabel("E - E0 (exact)")
ax_fidelity.legend()
plt.show()

**(b) (paper)** (1) For each $\delta$, the fidelity rises with $T$ and then levels off, lower for
larger $\delta$; with $\delta = 1$, a longer evolution does not help at all. Why? (Hint: notes,
section ["Adiabatic circuits"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-adiabatic-circuits): which Hamiltonian does a slow Trotter circuit follow?) (2) For
$\delta = 0.25$, how many Trotter steps does the first circuit with a fidelity above 0.99 have,
and how many CZs, with 2 CZs per bond gate? With an error of 0.5% per CZ, what is the probability
that it runs without an error? Compare with the 3 layers of `ansatz`, whose optimised angles of A2
reach a fidelity of 0.999.

*Your answer:*

**(c) (code, then paper)** On a device, we cannot compute the fidelity, and the energy comes from
shots. Repeat the energy plot with `measure_energy` and 1000 shots per setting, with the standard
errors as error bars: `plt.errorbar(x, y, yerr=...)`. The vertical axis is linear, since an
estimate can fall below $E_0$. Which conclusions of (a) can you still draw from the estimates?

In [ ]:
for delta in [0.25, 0.5, 1.0]:
    estimates = []                                # the estimate minus E0, one per total time
    standard_errors = []                          # its standard error, one per total time
    for total_time in total_times:
        steps = round(total_time / delta)
        # the estimate of measure_energy for the same circuit; append it minus E0, and its standard error
        # YOUR CODE HERE
    plt.errorbar(total_times, estimates, yerr=standard_errors, fmt="o-", capsize=3, label="δ = " + str(delta))
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("total time T")
plt.ylabel("E - E0 (estimate, 1000 shots per setting)")
plt.legend()
plt.show()

*Your answer:*

### Exercise 2 (core): gradient descent with the parameter-shift rule, and its cost in shots

Gradient descent steps downhill, $\theta \to \theta - \eta\,\nabla E(\theta)$, with a step size
$\eta$ (notes, section ["Optimising with shots"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-optimisers)). We estimate the gradient with the parameter-shift
rule (notes, section ["The parameter-shift rule"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-parameter-shift)),
$$
\frac{\partial E}{\partial\theta} = \frac{E(\theta + \frac{\pi}{2}) - E(\theta - \frac{\pi}{2})}{2} ,
$$
which holds for an angle that enters a single Pauli rotation. In the HVA, each angle enters
several gates: $\gamma_k$ all $n-1$ bond gates of layer $k$, and $\beta_k$ all its $n$ field
gates. By the product rule, the derivative with respect to a shared angle is the sum of the
derivatives with respect to the angle of each of its gates, so we shift one gate at a time.

For this we need a circuit with its own parameter for each gate. The given function
`hva_ansatz_per_gate` builds the same circuit as `hva_ansatz`, with one parameter per gate, and
also returns the list `angle_of_gate`: gate `gate` takes the HVA angle number `angle_of_gate[gate]`.
For a numpy array `angles` of HVA angles, `angles[angle_of_gate]` is then the array of the angles
of all the gates, in order. `gate_energy` does for this circuit what `energy_estimate` does for
`ansatz`.

In [ ]:
# The problem, defined again so that this exercise stands alone (as in Part A)
n_qubits = 4
hamiltonian = tfim(n_qubits, J, h)
E0 = np.linalg.eigvalsh(hamiltonian.to_matrix())[0]
n_layers = 3
ansatz = hva_ansatz(n_qubits, n_layers)
start = schedule(n_layers, 3.0)                     # the adiabatic schedule, total time 3

In [ ]:
def hva_ansatz_per_gate(n_qubits, n_layers):
    """The circuit of hva_ansatz(n_qubits, n_layers), with its own parameter for each gate.

    Also returns angle_of_gate: gate number gate takes the HVA angle angles[angle_of_gate[gate]]."""
    phi = ParameterVector("phi", n_layers * (2 * n_qubits - 1))
    circuit = QuantumCircuit(n_qubits)
    circuit.h(range(n_qubits))
    angle_of_gate = []
    gate = 0                                                   # the number of gates so far
    # the bonds (0,1), (2,3), ..., then (1,2), (3,4), ...
    bonds = list(range(0, n_qubits - 1, 2)) + list(range(1, n_qubits - 1, 2))
    for k in range(n_layers):
        for i in bonds:
            circuit.rzz(phi[gate], i, i + 1)
            angle_of_gate.append(2 * k)                     # gamma_k
            gate += 1
        for i in range(n_qubits):
            circuit.rx(phi[gate], i)
            angle_of_gate.append(2 * k + 1)                 # beta_k
            gate += 1
    return circuit, angle_of_gate


ansatz_per_gate, angle_of_gate = hva_ansatz_per_gate(n_qubits, n_layers)


def gate_energy(gate_angles, shots, history):
    """As energy_estimate, with one angle for each gate of ansatz_per_gate."""
    value, error = measure_energy(ansatz_per_gate.assign_parameters(gate_angles), shots, J, h)
    history.append(value)
    return value


print("angle_of_gate:", angle_of_gate)
print("the angle of each gate at the start:", np.round(start[angle_of_gate], 3))

**(a) (paper)** For our circuit ($n = 4$, $p = 3$), how many gates does one gradient shift, and
how many energy estimates and shots does it cost with 1000 shots per setting? Why can we not
shift $\gamma_k$ by $\pi/2$ in all its gates at once? (Hint: the bond gates of layer $k$ together
are $e^{-i\gamma_k P/2}$ with $P = \sum_i Z_iZ_{i+1}$. Which property of $P$ does the derivation
of the rule use?)

*Your answer:*

**(b) (code)** Complete `parameter_shift_gradient`. For each gate `gate`, the given lines copy
`gate_angles` into `plus` and `minus` (with `.copy()`, so that changing `plus` does not change
`gate_angles`). Shift the angle of gate `gate` by $\pm\pi/2$, and add half the difference of the two
estimates to the derivative of the HVA angle `angle_of_gate[gate]`. The cell after it compares your
gradient, with 100 000 shots per setting so that the shot noise is small, with the exact gradient. The exact
gradient is given; it uses finite differences of `energy_exact` with a tiny step, which are
accurate with exact energies (notes, section ["Finite differences"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-finite-differences)).

In [ ]:
def parameter_shift_gradient(angles, shots, history):
    """The gradient of the HVA energy at `angles`, by the parameter-shift rule, one gate at a time."""
    gate_angles = angles[angle_of_gate]                     # the angle of each gate
    gradient = np.zeros(len(angles))
    for gate in range(len(gate_angles)):
        plus = gate_angles.copy()
        minus = gate_angles.copy()
        # YOUR CODE HERE
    return gradient

In [ ]:
def exact_gradient(angles, step=1e-6):
    """The exact gradient, from finite differences of energy_exact (simulation only)."""
    gradient = np.zeros(len(angles))
    for j in range(len(angles)):
        plus = angles.copy()
        plus[j] += step
        minus = angles.copy()
        minus[j] -= step
        gradient[j] = (energy_exact(plus) - energy_exact(minus)) / (2 * step)
    return gradient


history = []
print("exact:          ", np.round(exact_gradient(start), 3))
print("parameter shift:", np.round(parameter_shift_gradient(start, 100_000, history), 3))
print("estimates used: ", len(history))

**(c) (code)** Complete `gradient_descent`. It makes `steps` steps from `angles` and records, after
each step, the exact energy of the new angles: only for our plots, since a device cannot give it.
Then run it from the schedule, with $\eta = 0.1$ and 1000 shots per setting. The plotting code is
given: `plt.semilogy` draws with a logarithmic vertical axis, and `np.array(energies) - E0`
subtracts $E_0$ from every entry.

Finally, change the call to use $\eta = 0.3$, and run the cell again. What happens, and why?
(Hint: along one direction, near the minimum, $E \approx E_{\min} + \frac{c}{2}(\theta - \theta^*)^2$.
By which factor does one step multiply the distance $\theta - \theta^*$?)

In [ ]:
def gradient_descent(angles, eta, steps, shots, history):
    """`steps` steps of gradient descent with step size eta, from `angles`.

    Returns the final angles, and the list of the exact energies after each step (simulation only)."""
    energies = []
    # YOUR CODE HERE
    return angles, energies


history = []
angles, energies = gradient_descent(start, 0.1, 30, 1000, history)
print("shots:", 2 * 1000 * len(history))
print("exact E - E0 after 30 steps:", round(energies[-1] - E0, 4))

plt.semilogy(range(1, 31), np.array(energies) - E0, "o-")
plt.xlabel("step")
plt.ylabel("E - E0 (exact)")
plt.show()

*Your answer:*

**What an optimisation costs in shots.** In the notes, the final accuracy of an optimisation is limited by the number of shots. Here we
measure the *total* number of shots that an optimisation spends, for COBYLA and for gradient
descent, with $N = 100$, 1000 and 10 000 shots per setting for every estimate. As the measure of
accuracy we use the exact energy of the angles found, minus $E_0$: a device cannot give it, but in
simulation it tells us how good the state really is.

**(d) (code)** Complete the loop. For each $N$, `cobyla_shots` is the number of shots of the COBYLA
run, and `cobyla_error` the exact energy at its final angles minus $E_0$; `gd_shots` and `gd_errors`
are arrays with one entry per step of gradient descent: the shots used up to that step, and the
exact energy after it minus $E_0$. Every step costs the same number of shots, and
`np.arange(1, steps + 1)` is the array $1, 2, \dots,$ `steps`. The plotting code is given:
`plt.loglog` draws with both axes logarithmic, and each $N$ gets its own colour, a line for
gradient descent and a square for COBYLA. (About 10 seconds.)

In [ ]:
steps = 30
for shots, colour in zip([100, 1000, 10_000], ["tab:blue", "tab:orange", "tab:green"]):
    cobyla_history = []
    result_N = minimize(energy_estimate, start, args=(shots, cobyla_history), method="COBYLA")
    gd_history = []
    angles_N, energies_N = gradient_descent(start, 0.1, steps, shots, gd_history)
    # cobyla_shots, cobyla_error: one number each; gd_shots, gd_errors: arrays, one entry per step
    # YOUR CODE HERE
    print("N =", shots, "  COBYLA:", cobyla_shots, "shots, E - E0 =", round(cobyla_error, 3),
          "  gradient descent:", int(gd_shots[-1]), "shots, E - E0 =", round(gd_errors[-1], 4))
    plt.loglog(gd_shots, gd_errors, "-", color=colour, label="gradient descent, N = " + str(shots))
    plt.loglog(cobyla_shots, cobyla_error, "s", color=colour, markersize=9, label="COBYLA, N = " + str(shots))

plt.xlabel("total shots")
plt.ylabel("E - E0 (exact)")
plt.legend(fontsize=8)
plt.show()

**(e) (paper)** From the plot and the printout: (1) How many shots does each optimiser need to get
below $E - E_0 = 0.05$, if it gets there at all? (2) Why does COBYLA stop long before reaching
$E_0$, even with $N = 10\,000$, while gradient descent keeps improving? (3) After 30 steps,
gradient descent ends at about the same error with $N = 1000$ and with $N = 10\,000$. What limits
it there, and what limits it with $N = 100$?

*Your answer:*

### Exercise 3 (core): find the mistakes

The cell below runs a VQE with shots, with both optimisers, and the report after it draws
conclusions from the output. The code runs without errors, but it contains **three conceptual
mistakes**, which make the report wrong: one in the COBYLA part, one in the gradient-descent part,
and one in the comparison of the two.

**Your task.** (a) Find each mistake, and explain in one or two sentences why it is wrong. (b) Fix
the code (in the empty cell after the report, starting from a copy of this one), run it, and
correct the report.

The mistakes are about what the numbers mean, not about Python. For each claim of the report, ask
how the number behind it was obtained, and what it cost. Change and re-run parts of the code to
test your guesses. The notes, sections ["The variational quantum eigensolver"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-vqe-loop) and ["Gradients from
shots"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-gradients), and Exercise 2 will help.

In [ ]:
# The problem, defined again so that this exercise stands alone (as in Part A)
n_qubits = 4
hamiltonian = tfim(n_qubits, J, h)
E0 = np.linalg.eigvalsh(hamiltonian.to_matrix())[0]
n_layers = 3
ansatz = hva_ansatz(n_qubits, n_layers)
start = schedule(n_layers, 3.0)                     # the adiabatic schedule, total time 3

In [ ]:
# A VQE with shots, and its report. This cell contains three conceptual mistakes.
shots = 1000
eta = 0.1                 # the step size of gradient descent
iterations = 20

# COBYLA
cobyla_history = []
cobyla_result = minimize(energy_estimate, start, args=(shots, cobyla_history), method="COBYLA")
cobyla_energy = cobyla_result.fun


# Gradient descent, with the gradient from finite differences: two estimates per angle, so 12 in
# all, instead of 42 for the parameter-shift rule. A small step makes the derivatives accurate.
def finite_difference_gradient(angles, shots, history, step=0.01):
    gradient = np.zeros(len(angles))
    for j in range(len(angles)):
        plus = angles.copy()
        plus[j] += step
        minus = angles.copy()
        minus[j] -= step
        gradient[j] = (energy_estimate(plus, shots, history) - energy_estimate(minus, shots, history)) / (2 * step)
    return gradient


gd_history = []
gd_angles = start.copy()
for iteration in range(iterations):
    gd_angles = gd_angles - eta * finite_difference_gradient(gd_angles, shots, gd_history)
gd_energy, gd_error = measure_energy(ansatz.assign_parameters(gd_angles), shots, J, h)

print("start:            E - E0 =", round(energy_exact(start) - E0, 3))
print("COBYLA:           E - E0 =", round(cobyla_energy - E0, 3), "  iterations:", cobyla_result.nfev)
print("gradient descent: E - E0 =", round(gd_energy - E0, 3), "±", round(gd_error, 3), "  iterations:", iterations)

**Report.**

1. *COBYLA.* COBYLA stopped when its trust region reached its lower bound. Its energy, printed
   above, is the energy of the state it found, up to the standard error of one estimate: about 0.05
   with 1000 shots per setting.
2. *Gradient descent.* Gradient descent did not converge: after 20 iterations, its energy is far
   above that of COBYLA, and often above the start. The step size `eta` = 0.1 is too large, so
   every step overshoots the minimum; a smaller `eta`, with more iterations, will fix this.
3. *Comparison.* Gradient descent needs fewer iterations than COBYLA (`iterations` = 20, against the
   number printed above), so once its step size is fixed, it is also the cheaper optimiser of the two.

*Your answer:*

In [ ]:
# Your corrected version of the cell above
# YOUR CODE HERE

### Exercise 4 (further): noise, and how many layers

Gate noise biases the energy. Depolarising noise pulls every expectation value towards zero
(Tutorial 2), so the energy rises towards that of the maximally mixed state,
$\mathrm{Tr}[H]/2^n = 0$. More layers make a better ansatz, but they add gates, and so noise: the
same trade-off as for the number of Trotter steps in Tutorial 3, Exercise 2.

We use the noise model and the density-matrix simulation of Tutorials 2 and 3 (copied below), with
the values of today's devices, $p = 0.1\%$ after every `r` gate and $0.5\%$ after every CZ. We
compute the noisy energy exactly, without shots, to see the bias alone: `noisy_density_matrix`
returns a density matrix $\rho$, and `rho.expectation_value(hamiltonian).real` is
$\mathrm{Tr}[\rho H]$. `circuit_energy` is `energy_exact` for any circuit, given as its second
argument.

In [ ]:
# The problem, defined again so that this exercise stands alone (as in Part A)
n_qubits = 4
hamiltonian = tfim(n_qubits, J, h)
E0 = np.linalg.eigvalsh(hamiltonian.to_matrix())[0]
n_layers = 3
ansatz = hva_ansatz(n_qubits, n_layers)
start = schedule(n_layers, 3.0)                     # the adiabatic schedule, total time 3
result_exact = minimize(energy_exact, start, method="COBYLA")   # exact energies, as in A2
print("HVA with exact energies: E - E0 =", round(result_exact.fun - E0, 4))

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error


def native_noise_model(p1, p2):
    """Depolarising errors after every r gate (p1) and every cz gate (p2), as in Tutorial 2, A2."""
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ["r"])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cz"])
    return noise_model


def noisy_density_matrix(circuit, noise_model):
    """Exact final density matrix of `circuit` (native gates, no measurements) under `noise_model`.

    From Tutorial 2, A3."""
    circuit = circuit.copy()
    circuit.save_density_matrix()
    density_simulator = AerSimulator(method="density_matrix", noise_model=noise_model)
    return density_simulator.run(circuit).result().data()["density_matrix"]


def circuit_energy(angles, circuit):
    """The exact energy of the state that `circuit` prepares with these angles (simulation only)."""
    return Statevector(circuit.assign_parameters(angles)).expectation_value(hamiltonian).real

**(a) (code, then paper)** Compile `ansatz` with the angles found with exact energies, `result_exact.x` (as in A2, computed
again in the first cell of this exercise), to `r`
and `cz` (with `optimization_level=1`, as in Tutorial 3), and print its gate counts. Then compute
its noisy energy.

From the gate counts, estimate as in Tutorial 2 the probability $f$ that the circuit runs without
an error, with the error probabilities $3p/4$ for an `r` gate and $15p/16$ for a CZ. If every error
turned the state into the maximally mixed state, the noisy energy would be $fE$: what bias
$E_{\text{noisy}} - E$ does this predict? Compare with the simulation.

In [ ]:
noise_model = native_noise_model(0.001, 0.005)
# native: the circuit with the angles result_exact.x, compiled to r and cz; print its gate counts.
# noisy_energy: Tr[rho H] for its noisy density matrix
# YOUR CODE HERE
print("without noise: E - E0 =", round(result_exact.fun - E0, 4))
print("with noise:    E - E0 =", round(noisy_energy - E0, 4))

*Your answer:*

**(b) (code, then paper)** For $p = 1, \dots, 5$ layers, minimise the exact energy with COBYLA,
starting from the schedule with steps of length 1, `schedule(layers, layers)`, and record the
optimised angles and the remaining error. `minimize` passes the circuit to `circuit_energy` with
`args=(circuit,)`: the comma makes `(circuit,)` a tuple with one element. Then complete
`noisy_errors`, which compiles each optimised circuit and returns its noisy energy minus $E_0$.
The plotting code is given; `plt.ylim(1e-3, 3)` sets the range of the vertical axis, so errors
below $10^{-3}$ are off the plot. Which number of layers gives the lowest noisy energy, with today's
noise? And with the noise of Tutorial 2, five times stronger, or with noise five times weaker than
today's?

In [ ]:
layer_numbers = [1, 2, 3, 4, 5]
optimised = []        # the optimised angles, one array per number of layers
ideal_errors = []     # the exact energy without noise minus E0, one per number of layers
for layers in layer_numbers:
    circuit = hva_ansatz(n_qubits, layers)
    # YOUR CODE HERE
print("without noise, E - E0:", np.round(ideal_errors, 4))


def noisy_errors(noise_model):
    """The noisy energy minus E0 at the optimised angles, one per number of layers."""
    errors = []
    # For each number of layers and its optimised angles, zip(layer_numbers, optimised): compile the
    # circuit to r and cz, and append Tr[rho H] - E0
    # YOUR CODE HERE
    return errors


today = noisy_errors(native_noise_model(0.001, 0.005))
print("today's noise, E - E0:", np.round(today, 3))
plt.semilogy(layer_numbers, ideal_errors, "o-", label="no noise")
plt.semilogy(layer_numbers, today, "s-", label="p = 0.1%, 0.5% (today)")
plt.semilogy(layer_numbers, noisy_errors(native_noise_model(0.005, 0.03)), "^-", label="p = 0.5%, 3% (Tutorial 2)")
plt.semilogy(layer_numbers, noisy_errors(native_noise_model(0.0002, 0.001)), "v-", label="p = 0.02%, 0.1%")
plt.ylim(1e-3, 3)
plt.xlabel("layers p")
plt.ylabel("E - E0 (exact, no shots)")
plt.legend()
plt.show()

*Your answer:*

### Exercise 5 (further): a hardware-efficient ansatz, and barren plateaus

The hardware-efficient ansatz of the notes (section ["Choosing an ansatz"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-ansatz)) applies, in each layer,
a PRX gate with its own two angles to every qubit, then CZ gates on the bonds
$(0,1), (2,3), \dots$, and then on $(1,2), (3,4), \dots$. In Qiskit, PRX$(\theta, \varphi)$ is
`circuit.r(theta, phi, qubit)` (Tutorial 2).

This exercise uses two given functions of earlier exercises: `circuit_energy`, from the second
code cell of Exercise 4, and in (b) `hva_ansatz_per_gate`, from the second code cell of Exercise 2.
If you skipped those exercises, run these two cells first.

In [ ]:
# The problem, defined again so that this exercise stands alone (as in Part A)
n_qubits = 4
hamiltonian = tfim(n_qubits, J, h)
eigenvalues, eigenvectors = np.linalg.eigh(hamiltonian.to_matrix())
E0 = eigenvalues[0]
ground_state = Statevector(eigenvectors[:, 0])     # the eigenvector of the lowest eigenvalue
n_layers = 3
ansatz = hva_ansatz(n_qubits, n_layers)
start = schedule(n_layers, 3.0)                     # the adiabatic schedule, total time 3
result_exact = minimize(energy_exact, start, method="COBYLA")   # exact energies, as in A2
print("E1 - E0 =", round(eigenvalues[1] - E0, 4))

**(a) (code, then paper)** Complete `hardware_efficient_ansatz`. Then minimise its exact energy with
COBYLA for 2 layers (16 angles), starting from small random angles,
`np.random.uniform(-0.1, 0.1, 16)` (angles close to zero start from a simple state, notes section
["Choosing an ansatz"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-ansatz)), with at most 1000 evaluations. The next cell compares the result with the
HVA of A2: the remaining error, the evaluations, and the fidelity with the ground state, `ground_state`
(from the first cell of this exercise).

The hardware-efficient ansatz ends close to $E_0 + 0.05$. The lowest state of the other parity
(notes, section ["Adiabatic state preparation"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#sec-adiabatic)) lies $E_1 - E_0 = 0.095$ above the ground state.
Using the notes' exercise ["How close is a low energy to the ground state?"](https://stefanopolla.github.io/ntqc/L4-ground-states.html#exr-variational), explain the fidelity
you find. Why does the HVA not have this problem?

In [ ]:
def hardware_efficient_ansatz(n_qubits, layers):
    """`layers` layers of PRX on every qubit (two angles each), then CZ on the bonds in two brick layers."""
    theta = ParameterVector("theta", 2 * n_qubits * layers)
    circuit = QuantumCircuit(n_qubits)
    j = 0                                        # the number of angles used so far
    # YOUR CODE HERE
    return circuit


hea = hardware_efficient_ansatz(n_qubits, 2)
print(hea.draw())

In [ ]:
from qiskit.quantum_info import state_fidelity

# result_hea: minimise circuit_energy for `hea` from small random angles, with options={"maxiter": 1000}
# YOUR CODE HERE

for name, circuit, res in [("HVA", ansatz, result_exact), ("hardware-efficient", hea, result_hea)]:
    state = Statevector(circuit.assign_parameters(res.x))
    print(name, ":  E - E0 =", round(res.fun - E0, 4), "  evaluations:", res.nfev,
          "  fidelity:", round(state_fidelity(state, ground_state), 3))

*Your answer:*

**(b) (code, then paper)** Barren plateaus. Complete `slope_spread`: for `samples` random choices of
all the angles (uniform between 0 and $2\pi$), compute the derivative of the energy with respect to
the first parameter, with the parameter-shift rule and exact energies, and return the standard
deviation of these derivatives (`np.std`). In both circuits below, the first parameter enters a
single gate: the $\theta$ of the first PRX of the hardware-efficient ansatz, and the angle of the
first bond gate of `hva_ansatz_per_gate`. This function is defined in the second code cell
of Exercise 2 (see the start of this exercise). The cell then computes the spread for chains of 2 to 10
qubits, with $4n$ layers, as in the figure of the notes (the figure uses the shared first angle of
the HVA instead). (About 10 seconds.)

By which factor does the spread of the hardware-efficient ansatz fall for every two qubits added,
from 6 qubits on? At that rate, how many more qubits make the typical slope 100 times smaller than
at 10 qubits, and how many more shots would it take to tell its sign?

In [ ]:
def slope_spread(circuit, ham, samples):
    """The standard deviation of dE/d(first parameter) of `circuit` at random angles, for the Hamiltonian `ham`."""
    slopes = []
    for _ in range(samples):
        angles = np.random.uniform(0, 2 * np.pi, circuit.num_parameters)
        # YOUR CODE HERE
    return np.std(slopes)


sizes = [2, 4, 6, 8, 10]
spread_hea, spread_hva = [], []
for size in sizes:
    ham = tfim(size, J, h)
    spread_hea.append(slope_spread(hardware_efficient_ansatz(size, 4 * size), ham, 100))
    circuit_per_gate, _ = hva_ansatz_per_gate(size, 4 * size)
    spread_hva.append(slope_spread(circuit_per_gate, ham, 100))
print("hardware-efficient:", np.round(spread_hea, 3))
print("HVA:               ", np.round(spread_hva, 3))

plt.semilogy(sizes, spread_hea, "s-", label="hardware-efficient")
plt.semilogy(sizes, spread_hva, "o-", label="HVA (first bond gate)")
plt.xlabel("qubits n")
plt.ylabel("typical slope (standard deviation)")
plt.legend()
plt.show()

*Your answer:*

## Wrap-up

- The adiabatic circuit prepares the ground state if its total time is long and its steps are
  short, but such circuits are deep. The HVA keeps its structure, with few layers, and lets an
  optimiser choose the angles.
- A parametrised circuit is built once, with `ParameterVector`, and `assign_parameters` fills in
  the angles; the energy is then a function of an array of angles, which scipy's `minimize` can
  optimise.
- With shots, the lowest estimate an optimiser has seen is biased low: estimate the energy at the
  final angles again, with fresh shots.
- COBYLA stops early once the shot noise hides the energy differences it compares. Gradient
  descent costs many more shots per step, but its noise averages out over the steps.
- Compare optimisers by the total number of shots they need for a given accuracy, not by their
  iterations.
- Gate noise raises the energy. More layers make a better ansatz but add noise, and the best number
  of layers balances the two.

Next week, we reduce the bias of Exercise 4 with error mitigation.